# آماده‌سازی و کاوش داده‌های فریب ترکی در ترکیه

این دفترچه فایل اکسل را می‌خواند، ساختار/کیفیت را بررسی می‌کند، چهار ستون جمله را به رکوردهای طولی تبدیل می‌کند و خروجی `data/deception_dataset.csv` می‌سازد. از ریشه پوشه پروژه اجرا کنید.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('data')
INPUT = DATA_DIR / 'deception_data.xlsx.xlsx'
OUTPUT = DATA_DIR / 'deception_dataset.csv'
if not INPUT.exists():
    raise FileNotFoundError(f'{INPUT} not found; run from project root.')


## بارگیری و بررسی ساختار

In [ ]:
xls = pd.ExcelFile(INPUT)
print('Sheets:', xls.sheet_names)
raw = pd.read_excel(INPUT, sheet_name=xls.sheet_names[0])
print('Shape:', raw.shape)
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
print('Missing values by column:\n', raw.isna().sum())
print('Duplicate rows:', raw.duplicated().sum())
print('Label types:\n', raw['Label_Type'].value_counts(dropna=False))
print('Target labels:\n', raw['Target_Label'].value_counts(dropna=False))

## پاکسازی و تبدیل به رکوردهای طولی
نگاشت برچسب‌ها مطابق دادهٔ ارزیابی اصلاح‌شده است: جمله‌های مستقیم مثبت و گزارش‌شده (`T_Dir_TR`, `FB_Dir_TR`) حقیقت‌نما؛ و صورت‌های منفی (`L_Dir_TR`, `L_Ind_TR`) فریبنده‌اند. ستون `Target_Label` اصلی نگه داشته می‌شود و به‌عنوان جایگزین قطعی برچسب زبانی استفاده نمی‌شود.

In [ ]:
source_to_truth = {'T_Dir_TR': 'Truthful', 'FB_Dir_TR': 'Truthful',
                    'L_Dir_TR': 'Deceptive', 'L_Ind_TR': 'Deceptive'}
required = ['ID', 'Subject_EN', 'Label_Type', *source_to_truth, 'Target_Label']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing expected columns: {missing}')
clean = raw.dropna(subset=required).copy()
for col in ['Subject_EN', 'Label_Type', *source_to_truth]:
    clean[col] = clean[col].astype(str).str.strip()
long = clean.melt(id_vars=['ID','Subject_EN','Label_Type','Target_Label'],
                  value_vars=list(source_to_truth), var_name='source_column',
                  value_name='sentence')
long['ground_truth'] = long['source_column'].map(source_to_truth)
long['sentence'] = long['sentence'].astype(str).str.strip()
long = long.rename(columns={'ID':'id','Subject_EN':'subject_en',
                            'Label_Type':'label_type','Target_Label':'target_label'})
long.insert(1, 'prompt_id', range(1, len(long)+1))
long = long[['id','prompt_id','sentence','ground_truth','label_type',
             'source_column','subject_en','target_label']]
long = long[long['sentence'].ne('')].reset_index(drop=True)
long['prompt_id'] = range(1, len(long)+1)
print('Prepared shape:', long.shape)
display(long.head(8))
print('Nulls:', int(long.isna().sum().sum()), '| Duplicate sentences:', int(long.duplicated(['sentence','source_column','id']).sum()))

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
long.to_csv(OUTPUT, index=False, encoding='utf-8-sig')
roundtrip = pd.read_csv(OUTPUT)
assert roundtrip.shape == long.shape
assert roundtrip['ground_truth'].isin(['Truthful','Deceptive']).all()
print(f'Wrote and verified {roundtrip.shape[0]} records to {OUTPUT}')